In [14]:
import folium
from folium import plugins 
import pandas as pd 
import random

#Loading data 
df = pd.read_csv(r'D:\Mission_Blitzkreig\Month_1\ooh_campaigns.csv')
df['roi'] = ((df['revenue'] - df['spend']) / df['spend'] * 100 ).round(2) 

# City co-ordinates 
city_coords ={
    'Dublin': (53.3498, -6.2603),
    'Cork': (51.8985, -8.4756),
    'Belfast': (54.5973, -5.9301),
    'Galway': (53.2707, -9.0568),
    'Limerick': (52.6638, -8.6267),
    'Waterford': (52.2593, -7.1101)
}

# Create billboard location
billboard_locations = df.copy()
latitudes = []
longitudes = []

for city in df['city'] :
    if city in city_coords:
        base_lat, base_lon = city_coords[city]
        lat = base_lat + random.uniform(-0.01, 0.01)
        lon = base_lon + random.uniform(-0.01, 0.01)
        latitudes.append(lat)
        longitudes.append(lon)
    else :
        latitudes.append(53.3498)  # Dublin-Default 
        longitudes.append(-6.2603)

billboard_locations['latitude'] = latitudes
billboard_locations['longitude'] = longitudes

print("Billboard Location Created!")

Billboard Location Created!


In [17]:
# Create Heat Map 
heat_map = folium.Map(
    location=[53.3498, -6.2603],
    zoom_start=6, 
    tiles='CartoDB positron'
    )
heat_data = []

for idx, row in billboard_locations.iterrows():
    heat_data.append([row['latitude'], row['longitude'], row['revenue']/1000])

plugins.HeatMap(
    heat_data,  # The data we prepared
    min_opacity=0.3,
    max_zoom=13,
    radius=25,
    blur=15,
    gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(heat_map)
heat_map.save('billboard_heat_map.html')
print("Heat Map Created!")
print("Heatmap density - red = high density, blue = low density")


Heat Map Created!
Heatmap density - red = high density, blue = low density


In [21]:
# Create map with marker clusters
cluster_map = folium.Map(
    location=[53.3498, -6.2603],
    zoom_start=6,
    tiles='CartoDB positron'
)

#Create marker cluster object 
marker_cluster = plugins.MarkerCluster(
    name='Billboard Locations',
    overlay=True,
    control=True,
    icon_create_function=None
).add_to(cluster_map)

#Add markers to the cluster
for idx, row in billboard_locations.iterrows():
    if row['roi'] > 100:
        color = 'green'
    elif row['roi'] > 50:
        color = 'orange'
    else:
        color = 'red'

    #create pop up text 
    popup_text = f"""
    <b>{row['campaign_name']}</b><br>
    City: {row['city']}<br>
    Format: {row['format']}<br>
    Revenue: €{row['revenue']:,.0f}<br>
    ROI: {row['roi']:.1f}%
    """

    # Add marker to cluster
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(marker_cluster)

# Save the map 
cluster_map.save('billboard_cluster_map.html')
print("Cluster map created")
print(" Zoom in to see individual markers, Zoom out to see clusters")



Cluster map created
 Zoom in to see individual markers, Zoom out to see clusters


In [27]:
# Create map with circle markers
circle_map = folium.Map(
    location=[53.4, -7.9],
    zoom_start=6,
    tiles='CartoDB positron'
)

# Add circle marker 
for idx , row in billboard_locations.iterrows():
    if row['roi'] > 100:
        color = 'green'
        fill_color = 'lightgreen'
    elif row['roi'] > 50:
        color ='orange'
        fill_color = 'lightyellow'
    else:
        color = 'red'
        fill_color = 'lightcoral'

popup_text = f"""
<b>{row['campaign_name']}</b><br>
    City: {row['city']}<br>
    Format: {row['format']}<br>
    Revenue: €{row['revenue']:,.0f}<br>
    ROI: {row['roi']:.1f}%
    """

# Add circle marker 
folium.CircleMarker(
    location=[row['latitude'], row['longitude']],
    radius=row['revenue']/1000,
    popup=folium.Popup(popup_text, max_width=250),
    color=color,
    fill=True,
    fill_color=fill_color,
    fill_opacity=0.6,
    weight=2
).add_to(circle_map)

# Save the Map
circle_map.save('billboard_circle_map.html')
print("Circle map created")
print("Circle size = revenue, color = ROI Performance")

Circle map created
Circle size = revenue, color = ROI Performance


In [31]:
# Create map with multiple layers
multi_layer_map = folium.Map(
    location=[53.4, -7.9],
    zoom_start=7,
    tiles='CartoDB positron'
)

# === LAYER 1: Regular Markers ===
marker_layer = folium.FeatureGroup(name='Markers')  # Create layer group first!

for idx, row in billboard_locations.iterrows():
    if row['roi'] > 100:
        color = 'green'
    elif row['roi'] > 50:
        color = 'orange'
    else:
        color = 'red'
    
    popup_text = f"""
    <b>{row['campaign_name']}</b><br>
    City: {row['city']}<br>
    Revenue: €{row['revenue']:,.0f}<br>
    ROI: {row['roi']:.1f}%
    """
    
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(marker_layer)

marker_layer.add_to(multi_layer_map)

# === LAYER 2: Circle Markers ===
circle_layer = folium.FeatureGroup(name='Revenue Circles')  # Create circle layer

for idx, row in billboard_locations.iterrows():
    if row['roi'] > 100:
        color = 'green'
        fill_color = 'lightgreen'
    elif row['roi'] > 50:
        color = 'orange'
        fill_color = 'lightyellow'
    else:
        color = 'red'
        fill_color = 'lightcoral'
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=row['revenue']/1000,
        color=color,
        fill=True,
        fill_color=fill_color,
        fill_opacity=0.6,
        weight=2,
        popup=f"{row['campaign_name']}: €{row['revenue']:,.0f}"
    ).add_to(circle_layer)

circle_layer.add_to(multi_layer_map)

# === LAYER 3: Heat Map ===
heat_layer = folium.FeatureGroup(name='Heat Map')  # Create heat layer

heat_data = [[row['latitude'], row['longitude'], row['revenue']/1000] 
             for idx, row in billboard_locations.iterrows()]

plugins.HeatMap(heat_data).add_to(heat_layer)
heat_layer.add_to(multi_layer_map)

# Add layer control
folium.LayerControl().add_to(multi_layer_map)

# Save map
multi_layer_map.save('billboard_multi_layer_map.html')
print("✅ Multi-layer map created: billboard_multi_layer_map.html")

✅ Multi-layer map created: billboard_multi_layer_map.html
